# Experiment 04 — Variable importance and interpretation

**Paper reference.** §2.2 / §3 of v15 — supports the qualitative narrative about prompt-induced behaviour.

**Goal.** For the Stage-2 linear model in each (scheme, policy), report (a) standardized coefficients, (b) drop-column importance (CV ΔMAE when each column is set to zero), and (c) permutation importance (CV ΔMAE when each column is shuffled). Use these to give a one-line natural-language interpretation per cell.

In [ ]:
import os, sys, warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import negolin_v15 as nv
nv.setup_matplotlib()

DATA_DIR = '/home/claude/data/merged_results'    # <-- edit if needed
OUT_DIR  = './out'
os.makedirs(OUT_DIR, exist_ok=True)

raw = nv.load_run_files(DATA_DIR)
_, turns = nv.preprocess_turns(raw)
POLICIES = list(nv.POLICIES)
SCHEMES  = list(nv.SCHEMES)
print(f'rows={len(turns):,}  policies={POLICIES}  schemes={SCHEMES}')

In [ ]:
# Load the Stage-2 selection produced by 02_main_sweep.ipynb
sel_path = Path(OUT_DIR) / 'selected_linear.csv'
if sel_path.exists():
    SELECTED = pd.read_csv(sel_path)
    print('Loaded selected_linear.csv:'); display(SELECTED.head())
else:
    # Sensible fallback: Feature II + Ridge per (scheme, policy)
    rows = [{'scheme': s, 'policy': p, 'chosen_family': 'II', 'chosen_model': 'Ridge'}
            for s in SCHEMES for p in POLICIES]
    SELECTED = pd.DataFrame(rows)
    print('selected_linear.csv not found -> using fallback (II/Ridge)')

def picked(scheme, policy):
    r = SELECTED.set_index(['scheme','policy']).loc[(scheme, policy)]
    fam = r['chosen_family']
    if fam == 'Base': fam = 'II'    # Base isn't a feature spec -- fall back
    return fam, r['chosen_model']

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.pipeline import Pipeline

def fit_and_coefs(scheme, policy):
    fam, mdl = picked(scheme, policy)
    sub = turns.query('scheme == @scheme and policy == @policy')
    meta, X, y, cols = nv.build_design_matrix(sub, fam)
    (_, _, _), (m_n, X_n, y_n) = nv.partition_persist(meta, X, y)
    if mdl == 'OLS':
        est = Pipeline([('s', StandardScaler()), ('m', LinearRegression())]).fit(X_n, y_n)
        a = float(est.named_steps['m'].intercept_)
    else:
        est = Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=1.0))]).fit(X_n, y_n)
        a = float(est.named_steps['m'].intercept_)
    coefs = dict(zip(cols, est.named_steps['m'].coef_))
    coefs['intercept'] = a
    return coefs, m_n, X_n, y_n, cols, est

# 24 rows of standardized coefficients
coef_rows = []
for sch in SCHEMES:
    for pol in POLICIES:
        coefs, *_ = fit_and_coefs(sch, pol)
        for k, v in coefs.items():
            coef_rows.append({'scheme': sch, 'policy': pol, 'feat': k, 'coef_std': float(v)})
coef_df = pd.DataFrame(coef_rows)
coef_df.to_csv(os.path.join(OUT_DIR, 'exp04_std_coefs.csv'), index=False)
coef_df.head(10).round(3)

## Coefficient heatmap (one matrix per scheme)

In [ ]:
# pick the most common chosen family per scheme to make a comparable heatmap
common_family = SELECTED.groupby('scheme')['chosen_family'].agg(
    lambda s: s.mode().iat[0]).to_dict()
print('most common family per scheme:', common_family)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, sch in zip(axes, SCHEMES):
    fam = common_family[sch]
    cols = nv.FEATURE_SPECS[fam]['cols']
    sub_coef = coef_df[coef_df['scheme'] == sch]
    mat = (sub_coef[sub_coef['feat'].isin(cols)]
             .pivot(index='feat', columns='policy', values='coef_std')
             .reindex(index=cols, columns=POLICIES))
    vmax = max(abs(mat.values.flatten()))
    im = ax.imshow(mat.values, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_title(f'scheme {sch} (family {fam})')
    ax.set_xticks(range(len(POLICIES))); ax.set_xticklabels(POLICIES, rotation=45)
    ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
    for i in range(len(cols)):
        for j in range(len(POLICIES)):
            v = mat.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:+.2f}', ha='center', va='center', fontsize=7,
                        color='white' if abs(v) > vmax/2 else 'black')
    fig.colorbar(im, ax=ax, fraction=0.04)
fig.suptitle('Standardized coefficients (only cells whose chosen family matches the column)', fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_exp04_coef_heatmap.png'))
plt.show()

## Drop-column and permutation importance for one example cell

In [ ]:
def drop_column_importance(meta, X, y, model_name='Ridge'):
    base_pred, _ = nv.grouped_cv_predict(meta, X, y, model_name)
    base_mae = nv.cv_metrics(base_pred)['mae']
    out = []
    for j in range(X.shape[1]):
        Xj = X.copy(); Xj[:, j] = 0.0
        pj, _ = nv.grouped_cv_predict(meta, Xj, y, model_name)
        out.append(nv.cv_metrics(pj)['mae'] - base_mae)
    return out, base_mae

def perm_importance(meta, X, y, model_name='Ridge', n_repeats=5, seed=0):
    base_pred, _ = nv.grouped_cv_predict(meta, X, y, model_name)
    base_mae = nv.cv_metrics(base_pred)['mae']
    rng = np.random.default_rng(seed)
    out = []
    for j in range(X.shape[1]):
        rep = []
        for _ in range(n_repeats):
            Xj = X.copy()
            Xj[:, j] = rng.permutation(Xj[:, j])
            pj, _ = nv.grouped_cv_predict(meta, Xj, y, model_name)
            rep.append(nv.cv_metrics(pj)['mae'])
        out.append(np.mean(rep) - base_mae)
    return out, base_mae

sch, pol = 'A', 'PVD'
_, m_n, X_n, y_n, cols, _ = fit_and_coefs(sch, pol)
drops, base_mae = drop_column_importance(m_n, X_n, y_n)
perms, _        = perm_importance(m_n, X_n, y_n, n_repeats=3)
importance = pd.DataFrame({
    'feat': cols, 'drop_col_dMAE': drops, 'permutation_dMAE': perms
}).sort_values('drop_col_dMAE', ascending=False)
print(f'baseline MAE={base_mae:.4f}'); display(importance.round(4))

**Interpretation cheatsheet** for the standardized coefficients:

- positive coef on `S^L_{t-1}` → seller's logit-state is sticky / mean-reverting weakly
- negative coef on `B_t` (or `dB_t`) → seller concedes when buyer offers more
- positive coef on `Y_{t-1}` → momentum (yesterday's concession predicts more concession today)
- positive coef on `B_t^-` (rare since B̃ ≥ 0) → would imply hostility to negative bids

Coefficients on the same feature should be roughly **stable across schemes within a policy** if the policy is a real underlying decision rule and not a scheme-specific artifact.